### Tools

Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code. Tools are pairing of: 

1. A schema, including the name of the tool, a description, and/or argument definitions (often a JSON schema) 
2. A function or coroutine to execute

In [1]:
## Importing a model and using it to get a response
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:qwen/qwen3-32b")
response = model.invoke("How can parrots talks?")
response 

AIMessage(content='<think>\nOkay, so the user is asking, "How can parrots talk?" Let me start by recalling what I know about parrots and their ability to mimic human speech. Parrots are known for their vocal mimicry, right? But why do they do that? I think it has something to do with the structure of their vocal organs.\n\nFirst, I should explain the anatomy. Parrots have a syrinx, which is their vocal organ. Unlike humans, who use vocal cords in the larynx, birds have the syrinx located at the base of the trachea. The syrinx might have two vocal chambers, allowing for more complex sounds. That could explain their ability to mimic a wide range of sounds, including human speech.\n\nThen there\'s the aspect of learning. I remember reading that parrots learn through imitation. They listen to the sounds around them and practice repeating them. But how exactly do they learn? Maybe it\'s similar to how humans learn to speak by listening to others. They might associate certain words with acti

In [2]:
## Creating a model for that tool
from langchain.tools import tool

## Here we will use it as decorator on a function, that will help the function to be recognized as a tool
@tool
def get_weather(location: str) -> str:
    """Get the weather information for a location"""
    return f"It's sunny in {location}."

## Now one method is that we create an agent using the create_agent function, or we can use the below method to bind the tool with the model
model_with_tools = model.bind_tools([get_weather])
model_with_tools

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.11'}}, output_version=None, profile={'name': 'Qwen3 32B', 'release_date': '2024-12-23', 'last_updated': '2024-12-23', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 40960, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x1180d1df0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x1181c1490>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'get_weather', 'description': 'Get the weather information for a location', 'par

In [ ]:
## Here in the response, you will see that the final response will not be generated. The content will be empty. The model just mention that it's going to use the tool.
response = model_with_tools.invoke("What is the weather of Boston today?")
response

AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for the weather in Boston today. I need to use the get_weather function. The function requires a location parameter. Boston is the location here. So I should call get_weather with location set to "Boston". Let me make sure there\'s no typo. Everything looks good. Let me format the tool call correctly.\n', 'tool_calls': [{'id': 'wmfbz59fb', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 93, 'prompt_tokens': 155, 'total_tokens': 248, 'completion_time': 0.14737141, 'completion_tokens_details': {'reasoning_tokens': 69}, 'prompt_time': 0.006146677, 'prompt_tokens_details': None, 'queue_time': 2.1491667469999998, 'total_time': 0.153518087}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_5cf921caa2', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provid

In [4]:
## By this we can see that what tools our model is using that will generate the response
for tool_call in response.tool_calls:
    print(f"Tools: {tool_call['name']}")
    print(f"Agrumnets: {tool_call['args']}")


Tools: get_weather
Agrumnets: {'location': 'Boston'}


#### Tool Execution Loop

As we know, with the bind tool method, we are not getting the final response, we have to create a complete loop of execution of tools that will generate the final response. BTW WE DON'T HAVE TO ALL THIS IF WE ARE USING THE CREATE_AGENT FUNCTION. IT WILL AUTOMATE ALL THESE PROCESS.

In [5]:
# 1. Model generates the tool calls
messages = [{"role": "user", "content": "What's weather in Boston?"}]
ai_message = model_with_tools.invoke(messages)
messages.append(ai_message)

# 2. Execute tools and collect results
for tool_call in ai_message.tool_calls:
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# 3. Pass result back to model to generate the final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)

The weather in Boston is sunny.
